# What's new in `unify-compilation-workflow`

A walkthrough of everything new on this branch — the unified
compilation workflow that turns AIE designs into Triton-style JIT
functions, packaged kernel factories, content-addressed caching,
tensor-arg validation, runtime/trace tooling, and the bench/verify
helpers that go with them.

Every cell here runs on a Ryzen AI NPU (Phoenix / NPU1 or Strix /
NPU2) with `ironenv` activated.  The device auto-detection in §0
picks whichever is attached.

**TL;DR — what's new:**

1. `@iron.jit` decorator — one Python function = one JIT'd design.
2. `aie.iron.kernels.*` factories — packaged C++ kernel wrappers (no more hand-rolled `Kernel(...)` + Makefile rules), now including the `bn_*` family used by MobileNet.
3. `aie.iron.algorithms.transform_parallel(...)` — multi-column elementwise dispatcher (`num_channels=1|2`, `pass_size_to_kernel`).
4. `Compile[T]` / `In` / `Out` / `InOut` annotation markers + 3 ways to bind compile params.
5. `CompilableDesign` / `CallableDesign` API: `as_mlir()` / `specialize()` / `compile()` / `__call__`.
6. Content-addressed kernel cache in `$NPU_CACHE_HOME` (default `~/.npu/cache/`), with hash invalidation on Peano upgrade *and* Peano-absent tolerance.
7. Tensor-arg validation against the lowered `aie.runtime_sequence` signature.
8. `aie.utils.benchmark` (`run_iters`, `print_benchmark`).
9. `aie.utils.verify` (`nearly_equal`, `count_mismatches`).
10. `aie.utils.trace` integration: TraceConfig + JIT + `trace_to_json` + `print_cycles_summary`.
11. Compile-process logging: every clang/aiecc command goes through `logging.getLogger("aie.utils.compile")`.
12. Cross-compilation (set the active device, build a binary for any NPU arch) plus per-arch kernel introspection (`kernels.mm.mac_dims`).
13. New `ObjectFifo` knobs: `aie_stream=(end, port)` for direct-stream output (kernels that emit via `put_ms()`) and `consumer_obj_type=` for asymmetric transfer granularity.
14. Cross-tile `Buffer` in `Worker.fn_args` — neighbor-tile L1 sharing now works without a Resolvable wrapper.
15. Lower-level IRON primitives (`Flow` / `Lock` / `TileDma` / `DmaChannel` / `Bd` / `Acquire` / `Release`) as peers of `ObjectFifo` for designs that hand-wire routing + DMA + sync.
16. `device_from_args(args, n_cols=...)` helper that collapses the `from_name(args.dev, n_cols=...)` boilerplate the example suite repeated ~30 times.

## 0. Setup

Pick a device and import the iron surface.

In [ ]:
import logging
import time
from pathlib import Path

import numpy as np

import aie.iron as iron
from aie.iron import (
    Compile,
    In,
    Out,
    ObjectFifo,
    Program,
    Runtime,
    Worker,
    jit,
    kernels,
)
from aie.iron.controlflow import range_

# No need to call set_current_device — iron.get_current_device() falls back
# to DefaultNPURuntime.device(), which queries XRT for the actual hardware
# (NPU1 = Phoenix, NPU2 = Strix). Both `lower()` and on-NPU calls use it.
device = iron.get_current_device()
print("Auto-detected device:", type(device).__name__)

## 1. `@iron.jit` — Triton-style design compilation

The decorator turns a Python function that *generates* an MLIR module
into something you call like a normal function. The body runs at
*compile* time; the returned `Program(...).resolve_program()` becomes
the cached kernel.

A minimal passthrough fits in 20 lines:

In [ ]:
@jit
def passthrough(x: In, y: Out, *, N: Compile[int]):
    line_ty = np.ndarray[(N,), np.dtype[np.uint8]]

    of_in = ObjectFifo(line_ty, name="in")
    of_out = ObjectFifo(line_ty, name="out")

    def core_fn(of_in, of_out):
        elem_in = of_in.acquire(1)
        elem_out = of_out.acquire(1)
        for i in range_(N):
            elem_out[i] = elem_in[i]
        of_in.release(1)
        of_out.release(1)

    worker = Worker(core_fn, [of_in.cons(), of_out.prod()])
    rt = Runtime()
    with rt.sequence(line_ty, line_ty) as (a_in, b_out):
        rt.start(worker)
        rt.fill(of_in.prod(), a_in)
        rt.drain(of_out.cons(), b_out, wait=True)
    return Program(iron.get_current_device(), rt).resolve_program()


# `passthrough` is now a CallableDesign, not a function.
print(repr(passthrough))

### Get the MLIR (cheap) — generate MLIR without the slow aiecc step

`as_mlir()` runs the generator and returns the MLIR module as a string.
Useful for sanity-checking the structure before the full compile.

In [ ]:
mlir = passthrough.as_mlir(N=4096)
print("\n".join(mlir.splitlines()[:6]))
print(f"... ({len(mlir.splitlines())} lines total)")

### Run it (compiles to xclbin on first call, then cached)

Calling the design with `iron.tensor` inputs JIT-compiles
(if not already cached) and runs on the NPU.

In [ ]:
N = 4096
in_t = iron.arange(N, dtype=np.uint8, device="npu")
out_t = iron.zeros(N, dtype=np.uint8, device="npu")

passthrough(in_t, out_t, N=N)
assert np.array_equal(in_t.numpy(), out_t.numpy())
print(f"PASS — {N} bytes round-tripped through the NPU.")

### New on this branch: extra `ObjectFifo` knobs

Two kwargs that landed during this round of design ports — both
opt-in, both eliminate Resolvable shims that designs were rolling
by hand.

**`aie_stream=(end, port)`** marks the fifo as a *direct AIE-stream*
connection (stamps `aie_stream` / `aie_stream_port` on the underlying
`aie.objectfifo` op).  Use it when a kernel emits each element via
`put_ms()` instead of acquire/release — the producer side gets no L1
buffer, the stream flows straight to the consumer.

**`consumer_obj_type=`** decouples producer-side and consumer-side
transfer granularity: the producer sends `obj_type`-sized chunks, the
consumer receives `consumer_obj_type`-sized chunks (producer element
count must be an integer multiple of consumer's).  Lets one DMA fan-out
serve consumers that want different chunk sizes without writing two
fifos and joining.

Below: emit MLIR for a tiny fifo declared with both, just to show the
attributes land on the lowered op.  No NPU run — the demo's purpose
is the API surface.

Worked example for `aie_stream`, taken from
`programming_examples/ml/magika/group2.py` (a compute tile streams its
output directly to the shim via `put_ms()`).  The relevant declaration:

```python
of_dout_L1L3 = ObjectFifo(dout_ty, name="of_dout_L1L3", depth=2, aie_stream=(0, 0))
```

lowers to (extract from `group2.py --emit-mlir`):

```mlir
aie.objectfifo @of_dout_L1L3(%logical_core, {%logical_shim_noc}, 2 : i32)
    {aie_stream = 0 : i32, aie_stream_port = 0 : i32}
    : !aie.objectfifo<memref<214xi32>>
```

— the `aie_stream` / `aie_stream_port` attributes ride on the same op as
a regular ObjectFifo declaration; the rest of the toolchain treats this
fifo as a wire-only connection (no L1 buffer, no acquire/release in the
core body).

`consumer_obj_type` example lives in
`test/python/objFifo_asymmetric.py` — producer sends 40-element chunks,
consumer receives 10-element chunks (4:1 ratio).  Lowered fifo type:

```mlir
aie.objectfifo @wts(...) : !aie.objectfifo<memref<40xi32>>
                        -> !aie.objectfifo<memref<10xi32>>
```

(The IRON consumer-side `acquire(1)` for asymmetric fifos is still
settling — track upstream if you hit a verifier complaint when wiring
the consumer through a `Worker.fn_args` handle.)

### Cross-tile `Buffer` in `Worker.fn_args`

AIE compute-tile cores can read their N/S/E/W *CoreTile* neighbors' L1
directly via shared memory.  `Worker` used to reject any `Buffer`
already pinned to a tile other than its own; now it honors the
explicit placement and `Program.resolve` picks up the neighbor tile
via `Buffer.tiles()`.

Pattern:

```python
neighbor = Tile(col=west_col, row=compute_row, tile_type=AIETileType.CoreTile)
lut_buf = Buffer(tile=neighbor, type=lut_ty, initial_value=lut_data, name="lut")

worker = Worker(
    body,
    fn_args=[..., lut_buf, ...],   # passed to kernel; resolves to memref
    tile=compute_tile,             # different from lut_buf.tile — that's fine
)
```

Useful when one compute tile needs more L1 than fits on its own tile
(e.g. magika group2 spreads 4 large LUTs across the compute tile and
its 3 neighbors).  The canonical demo is
`programming_examples/ml/magika/group2.py`.

A `Buffer` with no `tile=` set still auto-pins to the Worker's tile —
the convenience path is unchanged.

## 2. The kernel library: `aie.iron.kernels`

Before this branch, every design hand-rolled `Kernel("conv2dk1_i8",
"conv2dk1_i8.o", [int8_input_ty, ...])` declarations *and* maintained
a Makefile rule that compiled the `.cc` to `.o` with the right
`-D{INT8,UINT8}_ACT` flag.

The kernel library packages all of that — source path, compile flags,
typed argument list — behind one factory call.

In [ ]:
conv = kernels.conv2dk1(
    input_width=32,
    input_channels=64,
    output_channels=64,
    act_dtype=np.int8,
)
# `arg_types` is public.  The other introspection attributes
# (`_name`, `_source_file`, `_compile_flags`) are still private — we peek
# at them here purely to show what the factory packaged for you.
_src = conv._source_file
# Show the repo-relative path. The install layout mirrors the repo from
# `aie_kernels/...` onwards, so strip everything up to that prefix.
_repo_rel = "aie_kernels/" + _src.partition("aie_kernels/")[2]
print(f"function name : {conv._name}")
print(f"source        : {_repo_rel}")
print(f"compile flags : {conv._compile_flags}")
print(f"arg count     : {len(conv.arg_types())}")
for i, t in enumerate(conv.arg_types()[:3]):
    print(f"  arg[{i}] : {t}")
print("  ...")

The library covers conv variants, eltwise, reductions, activations
(ReLU, GeLU, SwiGLU, softmax), vision (filter2d, RGBA conversions),
matmul (`mm`, `mv`, `cascade_mm`), LUT-backed `bf16_exp`,
and bottleneck conv combos.  `mm` and `mv` expose a `.zero` sibling
Kernel for accumulator initialisation against the same `.o`;
`cascade_mm` exposes `.get_only`, `.put_only`, `.put_get`, and `.zero`
for the four bindings of one cascade chain:

In [ ]:
import aie.iron.kernels as K

named = sorted(n for n in dir(K) if not n.startswith("_") and not n[0].isupper())
factories = [
    n
    for n in named
    if n not in ("conv", "eltwise", "linalg", "reduce", "vision", "activation")
]
print(f"{len(factories)} factories available:")
for i in range(0, len(factories), 6):
    print("  " + ", ".join(factories[i : i + 6]))

### New on this branch: shared-buffer parameters

Two factories grew optional knobs for designs that share weight buffers
across multiple workers — both motivated by the resnet conv2_x port.

- **`kernels.conv2dk1_skip_init(skip_input_channels=...)`** — sizes the
  weights buffer for inline skip-projection weights and the skip buffer
  for the unprojected channel count.
- **`kernels.conv2dk3(weight_output_channels=...)`** — sizes the weights
  buffer for *more* output channels than a single call produces; multiple
  workers share one buffer and pick their slice via `channel_offset`.

In [ ]:
# Default: skip_input_channels falls back to input_channels.
default = kernels.conv2dk1_skip_init(input_channels=64, output_channels=256)
print(
    f"default (skip_in=64)   : wt={default.arg_shape(2)[0]:>6} bytes, "
    f"skip={default.arg_shape(4)[0]:>5} bytes"
)

# Explicit: skip path has narrower input (e.g. a lower-res branch).
narrow = kernels.conv2dk1_skip_init(
    input_channels=64,
    output_channels=256,
    skip_input_channels=32,
)
print(
    f"narrow  (skip_in=32)   : wt={narrow.arg_shape(2)[0]:>6} bytes, "
    f"skip={narrow.arg_shape(4)[0]:>5} bytes"
)

# Explicit: skip path is wider.
wide = kernels.conv2dk1_skip_init(
    input_channels=64,
    output_channels=256,
    skip_input_channels=128,
)
print(
    f"wide    (skip_in=128)  : wt={wide.arg_shape(2)[0]:>6} bytes, "
    f"skip={wide.arg_shape(4)[0]:>5} bytes"
)

print("\n  weights = (input_channels + skip_input_channels) * output_channels")
print("  skip    = input_width * skip_input_channels")

In [ ]:
default = kernels.conv2dk3(input_channels=64, output_channels=64)
shared = kernels.conv2dk3(
    input_channels=64,
    output_channels=32,
    weight_output_channels=64,
)

print(
    f"conv2dk3 defaults      : wt={default.arg_shape(3)[0]:>5}, out={default.arg_shape(4)[0]:>5}"
)
print(
    f"conv2dk3 shared-buffer : wt={shared.arg_shape(3)[0]:>5}, out={shared.arg_shape(4)[0]:>5}"
)
print("  (two workers share the 36864-byte weights, each writes 1024 bytes)")

### `transform_parallel` — multi-column elementwise dispatcher

For DDR-bandwidth-bound elementwise kernels (`relu`, `gelu`, `silu`,
`eltwise_add`, …), `aie.iron.algorithms.transform_parallel(...)`
distributes a 1-D tensor evenly across all NPU columns — and, with
`num_channels=2`, both shim DMA channels per column.  Drops the need
for the design to know how many columns the device has.

```python
from aie.iron.algorithms import transform_parallel

@iron.jit
def my_relu(input: In, output: Out, *, N: Compile[int], dtype: Compile[type]):
    return transform_parallel(
        input, output,
        func=kernels.relu(...),
        N=N, dtype=dtype,
        tile_size=16,
        num_channels=2,           # use both shim channels per column
        pass_size_to_kernel=True, # kernel signature: (in, out, n)
    )
```

`pass_size_to_kernel=False` for `(in, out)`-only kernel signatures.
See `programming_examples/ml/eltwise_unary/eltwise_unary.py` for the
end-to-end design that selects between `relu` / `silu` / `gelu` via
a `Compile[str]` knob.

## 3. `Compile[T]` parameters: three flavours

Compile-time parameters use `Compile[type]` and **must be keyword-only**
(place a `*,` before them). Three ways to bind them, mirroring Triton:

In [ ]:
@jit  # A: bare decorator, params at call time
def gemm_a(a: In, b: In, c: Out, *, M: Compile[int], N: Compile[int]):
    pass


@jit(M=512, N=512)  # B: pre-bound params at decoration
def gemm_b(a: In, b: In, c: Out, *, M: Compile[int], N: Compile[int]):
    pass


@jit  # C: defaults in signature
def gemm_c(a: In, b: In, c: Out, *, M: Compile[int] = 256, N: Compile[int] = 256):
    pass


print("a:", gemm_a.compilable.compile_kwargs)
print("b:", gemm_b.compilable.compile_kwargs)
print("c:", gemm_c.compilable.compile_kwargs, "(uses signature defaults at lower-time)")

Unknown kwargs to `@iron.jit` raise `TypeError` immediately at decoration
time — fails fast so typos don't silently run a kernel with no value bound.
Non-keyword-only `Compile[T]` params raise `TypeError` with a fix-it
suggestion.

In [ ]:
try:

    @jit(NOPE=99)
    def oops(x: In, y: Out, *, N: Compile[int]):
        pass

except TypeError as e:
    print("TypeError:", e)

## 4. `as_mlir()` vs `specialize()` vs `compile()` vs `__call__`

A `CallableDesign` exposes four entry points with very different costs.

| call            | does                                          | cost      |
|-----------------|-----------------------------------------------|-----------|
| `.specialize(**kw)` | bind compile params, return new CallableDesign | µs        |
| `.as_mlir(**kw)` | run generator → MLIR string                  | ms        |
| `.compile()`    | full aiecc → cached `final.xclbin` + `insts.bin` | s (then cached) |
| `design(*tensors, **kw)` | compile (cached) + run on NPU       | ms after first |


In [ ]:
t = time.perf_counter()
spec = passthrough.specialize(N=4096)
print(f"specialize()  : {(time.perf_counter() - t) * 1e6:6.0f} µs")

t = time.perf_counter()
mlir = spec.as_mlir()
print(
    f"as_mlir()     : {(time.perf_counter() - t) * 1e3:6.1f} ms  ({len(mlir):,} chars)"
)

t = time.perf_counter()
xclbin, insts = spec.compile()
print(f"compile()     : {(time.perf_counter() - t) * 1e3:6.1f} ms  (cache hit)")
print(f"  xclbin -> {xclbin}")
print(f"  insts  -> {insts}")

### Progressive binding: chained `.specialize()`

`specialize()` is **immutable-style** — it returns a *new* `CallableDesign`
with the kwargs merged in, leaving the original unchanged. That makes it
trivial to bind compile params progressively: pin one now, pin the rest
later, override anything along the way. Standard `functools.partial` also
works on `.as_mlir` / `.compile` / `.__call__` since they're plain methods —
no JIT machinery in the way.


In [ ]:
import functools


@jit
def add_const(x: In, y: Out, *, N: Compile[int], dtype: Compile[type]):
    line_ty = np.ndarray[(N,), np.dtype[dtype]]
    of_in = ObjectFifo(line_ty, name="in")
    of_out = ObjectFifo(line_ty, name="out")

    def core_fn(of_in, of_out):
        elem_in = of_in.acquire(1)
        elem_out = of_out.acquire(1)
        for i in range_(N):
            elem_out[i] = elem_in[i] + 1
        of_in.release(1)
        of_out.release(1)

    worker = Worker(core_fn, [of_in.cons(), of_out.prod()])
    rt = Runtime()
    with rt.sequence(line_ty, line_ty) as (a_in, b_out):
        rt.start(worker)
        rt.fill(of_in.prod(), a_in)
        rt.drain(of_out.cons(), b_out, wait=True)
    return Program(iron.get_current_device(), rt).resolve_program()


# Chain: bind N now, bind dtype later. Each call returns a fresh design.
half = add_const.specialize(N=4096)
full = half.specialize(dtype=np.int32)
print("half:    ", repr(half))
print("full:    ", repr(full))
print("original:", repr(add_const), "  # untouched")

# Override: re-specialize the same key.
wider = full.specialize(N=8192)
print("wider:   ", repr(wider))

# functools.partial works too — .as_mlir is just a method.
as_mlir_N4096 = functools.partial(add_const.as_mlir, N=4096)
mlir = as_mlir_N4096(dtype=np.int32)
print(f"\nfunctools.partial(.as_mlir, N=4096)(dtype=int32) -> {len(mlir):,} chars MLIR")

### Peek at what `compile()` actually runs

All compile-path subprocesses (`clang++` for ExternalFunctions, `aiecc`
for the design itself) go through `logging.getLogger("aie.utils.compile")`.
Bumping it to `DEBUG` reveals every command line plus cache-hit/miss
messages — enough to grab the exact `clang++` invocation aiecc handed
off, paste it into a terminal, and iterate. Force a fresh compile with
`use_cache=False` so you can see every clang/aiecc invocation light up:

In [ ]:
logging.basicConfig(level=logging.WARNING, format="%(name)s | %(message)s", force=True)
logging.getLogger("aie.utils.compile").setLevel(logging.DEBUG)


# `use_cache=False` forces a fresh compile; using kernels.passthrough means
# you also see the per-ExternalFunction `clang++` invocation, not just the
# aiecc subprocess. Both lines come from the same logger.
@jit(use_cache=False)
def passthrough_logged(x: In, y: Out, *, N: Compile[int]):
    line_size = N // 4
    line_ty = np.ndarray[(line_size,), np.dtype[np.uint8]]
    of_in = ObjectFifo(line_ty, name="in")
    of_out = ObjectFifo(line_ty, name="out")
    pt = kernels.passthrough(tile_size=line_size, dtype=np.uint8)

    def core_fn(of_in, of_out, pt):
        elem_in = of_in.acquire(1)
        elem_out = of_out.acquire(1)
        pt(elem_in, elem_out, line_size)
        of_in.release(1)
        of_out.release(1)

    worker = Worker(core_fn, [of_in.cons(), of_out.prod(), pt])
    rt = Runtime()
    with rt.sequence(line_ty, line_ty) as (a_in, b_out):
        rt.start(worker)
        rt.fill(of_in.prod(), a_in)
        rt.drain(of_out.cons(), b_out, wait=True)
    return Program(iron.get_current_device(), rt).resolve_program()


_ = passthrough_logged.specialize(N=4096).compile()

# Quiet the logger again so the rest of the walkthrough isn't noisy.
logging.getLogger("aie.utils.compile").setLevel(logging.WARNING)

### Passing custom flags to `aiecc`

`@iron.jit(aiecc_flags=[...])` forwards extra flags straight to the
underlying `aiecc` invocation.  Use it when the design needs a non-
default compile mode the kernel-library helpers can't express.

Common one: the AIE buffer allocator picks a bank-aware scheme by
default; some designs (matmul whole_array, cascade) want
`basic-sequential` (linear allocation) for predictable layout, which
is enough to swing perf by 25%+ on memory-pressured matmuls.


In [ ]:
@iron.jit(aiecc_flags=["--alloc-scheme=basic-sequential"])
def alloc_scheme_demo(a_in: In, b_out: Out, *, N: Compile[int]):
    line_ty = np.ndarray[(N,), np.dtype[np.uint8]]
    of_in = ObjectFifo(line_ty, name="af_in")
    of_out = ObjectFifo(line_ty, name="af_out")

    def core_fn(of_in, of_out):
        elem_in = of_in.acquire(1)
        elem_out = of_out.acquire(1)
        for i in range_(N):
            elem_out[i] = elem_in[i]
        of_in.release(1)
        of_out.release(1)

    worker = Worker(core_fn, [of_in.cons(), of_out.prod()])
    rt = Runtime()
    with rt.sequence(line_ty, line_ty) as (a, b):
        rt.start(worker)
        rt.fill(of_in.prod(), a)
        rt.drain(of_out.cons(), b, wait=True)
    return Program(iron.get_current_device(), rt).resolve_program()


# The flag travels through `.specialize().compile()` into aiecc.
xclbin, _insts = alloc_scheme_demo.specialize(N=4096).compile()
print(f"compiled with --alloc-scheme=basic-sequential: {xclbin.name}")

## 5. Content-addressed caching

Every compiled design lands in `$NPU_CACHE_HOME/<hash>/` (default
`~/.npu/cache/<hash>/` if `NPU_CACHE_HOME` is unset). The hash is
content-addressed on:

- the generator function's bytecode + closure
- the resolved `compile_kwargs` (sorted, JSON-stable)
- the target arch (`aie2` / `aie2p`)
- Peano's `clang++` mtime (so a Peano upgrade auto-invalidates)
- aiecc's mtime
- any `source_files` mtimes (so a `.cc` edit auto-invalidates)

Re-running an unchanged design is ~free; changing *anything* triggers
a fresh compile.

Each per-design directory keeps **every artifact** aiecc produced —
useful when you need to inspect the lowered MLIR, the post-placement IR,
the per-core LLVM IR, or the actual `.o` files that linked into the
final xclbin:

In [ ]:
from aie.utils.compile import NPU_CACHE_HOME

# What each artifact in the per-design cache dir is. (`xx` matches whichever
# row/col the AIE core landed on, e.g. `main_core_0_2.elf`.)
_ARTIFACT_DESC = {
    ".lock": "cross-process flock so concurrent compiles don't collide",
    "aie.mlir": "input MLIR your generator produced (post-resolve_program)",
    "input_with_addresses.mlir": "lowered MLIR with shim/memtile DMA addresses bound",
    "final.xclbin": "the binary XRT loads onto the NPU",
    "insts.bin": "NPU runtime instruction stream (paired with the xclbin)",
    "main.pdi": "Programmable Device Image — config data packed by bootgen",
    "main_aie_partition.json": "AIE partition manifest (which tiles, kernels, memory)",
    "main_aie_cdo_elfs.bin": "CDO blob carrying the per-core ELFs to the device",
    "main_aie_cdo_enable.bin": "CDO blob enabling tiles after init",
    "main_aie_cdo_init.bin": "CDO blob initialising tile/DMA registers",
    "main_design.bif": "Bootgen Image Format spec used to assemble main.pdi",
    "main_kernels.json": "kernel metadata that XRT consumes alongside the xclbin",
    "main_mem_topology.json": "memory-topology descriptor embedded into the xclbin",
    "main_core_xx.elf": "per-core final ELF the AIE tile actually runs",
    "main_core_xx.ld.script": "linker script Peano used to lay out that ELF",
    "main_core_xx.ll": "per-core LLVM IR before opt",
    "main_core_xx.opt.ll": "per-core LLVM IR after opt",
    "main_core_xx.peanohack.ll": "Peano workaround IR (vector-intrinsic fixups)",
    "main_core_xx.o": "per-core compiled object that links into the .elf",
    "passThroughLine.cc": "ExternalFunction kernel source, copied in for clang",
    "passThroughLine.o": "clang's compiled .o for that ExternalFunction",
}


def _describe(name: str) -> str:
    """Look up an artifact, normalising per-core filenames to `xx`."""
    if name in _ARTIFACT_DESC:
        return _ARTIFACT_DESC[name]
    import re

    return _ARTIFACT_DESC.get(re.sub(r"_\d+_\d+", "_xx", name), "")


print(f"Cache root: {NPU_CACHE_HOME}")
print(f"Cache entries on this machine: {len(list(NPU_CACHE_HOME.glob('*')))}")
kernel_dir = Path(xclbin).parent
print(f"\npassthrough(N=4096) lives at: {kernel_dir}\n")
for p in sorted(kernel_dir.iterdir()):
    print(f"  {p.name:<30} {_describe(p.name)}")

## 6. Tensor-arg validation

When you call a JIT design, the framework reads the host-facing
`aie.runtime_sequence`'s typed arguments straight from the lowered MLIR
and checks each runtime tensor's element count against the declared
memref size. The runtime_sequence signature *is* the host-side contract,
so there's no need to walk DMA ops or fold multi-DMA patterns back
together — fan-outs, repeated loads, and InOut fill+drain pairs all
resolve correctly because the arg type doesn't change.

Modules with multiple runtime_sequences (helper sub-sequences invoked
via `aiex.run`, or per-device sequences in a multi-device program) are
handled by picking the unique **call-graph root** — the sequence no
other sequence calls. If there's no unique root (multi-device with
several roots, or an unnamed sequence we can't reason about) validation
is skipped rather than guessing wrong.

Validation runs on both fresh compiles and cache hits, so wrong-size
tensors fail loudly *before* the NPU runs:

In [ ]:
wrong_size = iron.zeros(99, dtype=np.uint8)
try:
    passthrough(wrong_size, out_t, N=4096)
except RuntimeError as e:
    print("Caught expected error:")
    print(f"  {e}")

In [ ]:
# `parse_dma_sizes` is a private helper exposed here just so we can peek
# at what validation expects. After a `compile()` (or any call), the same
# list is also stored on the design at `compilable._expected_tensor_sizes`.
# It reads the entry-point runtime_sequence's typed memref args directly,
# so this matches the host-side contract aiecc generated for the design.
from aie.utils.compile.jit._dma_size_parser import parse_dma_sizes
from aie.iron import In, Out, InOut
import inspect

cache_dir = Path(xclbin).parent
sizes = parse_dma_sizes(cache_dir)

# Direction (In / Out / InOut) lives on the @iron.jit generator's
# signature, not in the runtime_sequence MLIR — re-introspect the original
# generator to pair each arg's element count with its declared kind.
_KIND = {In: "In", Out: "Out", InOut: "InOut"}
sig = inspect.signature(passthrough.compilable.mlir_generator)
tensor_params = [
    (name, _KIND[p.annotation])
    for name, p in sig.parameters.items()
    if p.annotation in _KIND
]

print(f"runtime_sequence arg sizes for passthrough(N=4096): {sizes}")
for i, (size, (name, kind)) in enumerate(zip(sizes, tensor_params)):
    print(f"  arg[{i}] {name!r:8} {kind:5} : {size} elements")

## 7. Helper utilities cheat-sheet

A round-up of the helpers that landed on this branch — what to import
and when to reach for them.

| Helper | What it does |
|--------|--------------|
| `aie.iron.tensor / zeros / ones / rand / randint / arange` | Build host tensors targeting `device="npu"` (or `"cpu"`). |
| `aie.iron.{In, Out, InOut, Compile}` | Type annotation markers for `@jit` generator params. |
| `aie.iron.{jit, CompilableDesign, CallableDesign}` | The JIT decorator + the two design wrapper classes. |
| `aie.iron.kernels.*` | Pre-packaged kernel factories — see §2. |
| `aie.iron.{Buffer, Lock, Flow, TileDma, DmaChannel, Bd, Acquire, Release}` | Lower-level structural primitives — peers of `ObjectFifo` — see §12. |
| `aie.iron.algorithms.transform_parallel` | Multi-column elementwise dispatcher — see §2. |
| `aie.iron.device.device_from_args(args, n_cols=...)` | Resolve an argparse Namespace to a `Device` without the `from_name(args.dev, n_cols=...)` boilerplate (with an `n_cols="auto"` mode that reads `args.n_cols` if present). |
| `aie.iron.{compile_context, get_compile_arg}` | Inject compile-time values into generators that aren't kwargs of the top-level function (advanced). |
| `aie.utils.benchmark.{run_iters, print_benchmark}` | Warmup + timed iters → stats. |
| `aie.utils.verify.{nearly_equal, count_mismatches}` | Sloppy-equality compare for LUT/saturating kernel outputs. |
| `aie.utils.trace.TraceConfig` | Hardware trace plumbing — see §8. |
| `aie.utils.trace.utils.print_cycles_summary` | Pretty-print per-event cycle counts from a parsed trace. |
| `aie.utils.compile.NPU_CACHE_HOME` | Cache root path (default `~/.npu/cache`; override with `$NPU_CACHE_HOME`). |
| `aie.utils.compile.jit._dma_size_parser.parse_dma_sizes` | Per-host-arg element counts read from the entry-point `aie.runtime_sequence`'s typed args. |
| `aie.utils.hostruntime.set_current_device` | Set the device used by `iron.get_current_device()` and JIT compile. |
| `aie.utils.DefaultNPURuntime` | Cached XRT runtime — auto-picks NPU1/NPU2 from the device's product name. |

**Reading the docstrings.** Every helper above has one.
Three ways to read it without leaving the notebook:

* `help(some_helper)` — Python built-in, prints signature + docstring.
* `some_helper?` — Jupyter / IPython shortcut, opens a side panel.
* `print(some_helper.__doc__)` — raw access, works everywhere.

Worked example:

In [ ]:
from aie.utils.verify import count_mismatches

help(count_mismatches)

In [ ]:
# `compile_context` — inject compile args into nested helpers
from aie.iron import compile_context, get_compile_arg


def make_fifo_pair(line_ty):
    # This helper doesn't take a `name_prefix` arg, but it can read one
    # from the active CompileContext.
    prefix = get_compile_arg("prefix", default="")
    return ObjectFifo(line_ty, name=f"{prefix}in"), ObjectFifo(
        line_ty, name=f"{prefix}out"
    )


with compile_context(prefix="layer1_"):
    line_ty = np.ndarray[(64,), np.dtype[np.uint8]]
    of_in, of_out = make_fifo_pair(line_ty)
    print("Inside context:", of_in.name, of_out.name)

# Outside the context, the default kicks in.
of_in, of_out = make_fifo_pair(line_ty)
print("Outside context:", of_in.name, of_out.name)

## 8. Hardware trace via `TraceConfig`

The hardware trace machinery threads through `@iron.jit` in two steps:

1. **Generator side**: declare `trace_config: Compile[TraceConfig | None] = None`,
   wrap your `Worker(...)` with `trace=1 if trace_config else 0`,
   and call `rt.enable_trace(trace_config.trace_size, workers=[...])`
   inside the runtime sequence.

2. **Caller side**: pass `trace_config=TraceConfig(trace_size=N)` at
   call time. After the call, `trace_config.physical_mlir_path` is
   populated so you can run `trace_config.trace_to_json(...)` to parse
   `trace.txt` into a Chrome-tracing JSON.

In [ ]:
from aie.utils.trace import TraceConfig


@jit
def passthrough_with_trace(
    x: In,
    y: Out,
    *,
    N: Compile[int],
    trace_config: Compile[TraceConfig | None] = None,
):
    line_size = N // 4
    tensor_ty = np.ndarray[(N,), np.dtype[np.uint8]]  # what the host hands in
    line_ty = np.ndarray[(line_size,), np.dtype[np.uint8]]  # per-tile chunks
    of_in = ObjectFifo(line_ty, name="in")
    of_out = ObjectFifo(line_ty, name="out")

    # Use the kernel-library passthrough — its C++ body calls event0() at
    # entry and event1() at exit, which is what `print_cycles_summary`
    # below pairs up to count per-kernel cycles. An inline Python copy loop
    # would still produce a real trace but with no event0/event1 pairs, so
    # the summary would report 0 kernel invocations.
    pt = kernels.passthrough(tile_size=line_size, dtype=np.uint8)

    def core_fn(of_in, of_out, pt):
        for _ in range_(4):  # 4 line tiles per N-element invocation
            elem_in = of_in.acquire(1)
            elem_out = of_out.acquire(1)
            pt(elem_in, elem_out, line_size)
            of_in.release(1)
            of_out.release(1)

    # Wire trace=1 onto each worker we want to instrument.
    worker = Worker(
        core_fn, [of_in.cons(), of_out.prod(), pt], trace=1 if trace_config else 0
    )

    rt = Runtime()
    with rt.sequence(tensor_ty, tensor_ty) as (a_in, b_out):
        if trace_config:
            rt.enable_trace(trace_config.trace_size, workers=[worker])
        rt.start(worker)
        rt.fill(of_in.prod(), a_in)
        rt.drain(of_out.cons(), b_out, wait=True)
    return Program(iron.get_current_device(), rt).resolve_program()


# `as_mlir()` runs the generator and shows the MLIR — confirm the trace
# wiring (`packetflow`s into the trace ddr buffer) was actually emitted.
trace_cfg = TraceConfig(trace_size=8192)
mlir_with_trace = passthrough_with_trace.as_mlir(N=4096, trace_config=trace_cfg)
trace_lines = [
    l for l in mlir_with_trace.splitlines() if "trace" in l.lower() or "packetflow" in l
]
print(f"{len(trace_lines)} trace-related ops in the lowered MLIR:")
for line in trace_lines[:6]:
    print("  " + line.strip()[:100])

**Calling it on hardware** writes a `trace.txt` next to your cwd.
`trace_config.physical_mlir_path` is auto-populated after the call so
`trace_config.trace_to_json(physical_mlir_path, output_name)` parses the
raw trace into a Chrome-tracing JSON, and
`aie.utils.trace.utils.print_cycles_summary` gives you the per-event
cycle breakdown — useful for spotting which kernel call is the
bottleneck. The runnable end-to-end driver also lives in
`programming_examples/basic/passthrough_kernel/passthrough_kernel.py`
(or `make trace`). Inline demo using the design from the cell above:

In [ ]:
import os
import json
from aie.utils.trace.utils import print_cycles_summary

trace_cfg = TraceConfig(trace_size=8192)
in_t = iron.arange(N, dtype=np.uint8, device="npu")
out_t = iron.zeros(N, dtype=np.uint8, device="npu")

# Hardware run with trace=on. trace.txt lands in cwd; physical_mlir_path
# gets populated on the cfg so we can parse against the lowered IR.
passthrough_with_trace(in_t, out_t, N=N, trace_config=trace_cfg)
print(f"trace.txt        : {os.path.getsize('trace.txt')} bytes")
# `print(trace_cfg)` uses TraceConfig.__str__ — eval-faithful repr first,
# then any post-run state (physical_mlir_path, last_tensor_*) indented
# underneath. `repr(trace_cfg)` would just be the constructor-args line.
print(trace_cfg)

# Parse to Chrome-tracing JSON (open in chrome://tracing or perfetto).
trace_cfg.trace_to_json(trace_cfg.physical_mlir_path, "trace_demo.json")
events = json.load(open("trace_demo.json"))
print(f"trace_demo.json  : {len(events)} events")

# print_cycles_summary walks the JSON and reports per-tile / per-event
# cycle counts. For this single-iter passthrough it just confirms the
# trace landed on the expected core; richer designs surface kernel-level
# cycle hot-spots here.
print()
print_cycles_summary("trace_demo.json")

## 9. `aie.utils.benchmark` — timing + warmup

`run_iters(fn, *args, warmup=N, iters=M)` runs the design `M+N` times
(discarding the first `N`) and returns a `BenchmarkResult` with
`avg_us` / `min_us` / `max_us` for both end-to-end host latency and
NPU-only time.  `print_benchmark(result)` formats it nicely.

In [ ]:
from aie.utils.benchmark import run_iters, print_benchmark

bench = run_iters(passthrough, in_t, out_t, N=4096, warmup=2, iters=10)
print_benchmark(bench)

## 10. `aie.utils.verify` — sloppy equality for AIE outputs

NPU kernels are often LUT approximations or use saturating arithmetic,
so exact `np.array_equal` is too strict. `count_mismatches(actual, ref,
rtol=0.128, atol=None)` returns `(n_errors, n_checked)` — defaults
to 12.8% relative tolerance and stops at the first inf / nan.
`nearly_equal(...)` returns the boolean mask if you need it.

In [ ]:
from aie.utils.verify import count_mismatches, nearly_equal

rng = np.random.default_rng(seed=0)  # deterministic output for the docs
ref = np.linspace(0, 10, 1000, dtype=np.float32)
approx = ref + rng.uniform(-0.05, 0.05, size=ref.shape).astype(np.float32)
errors, n = count_mismatches(approx, ref, rtol=0.05)
print(f"count_mismatches(rtol=0.05): {errors} / {n} samples outside tolerance")

mask = nearly_equal(approx, ref, rtol=0.05)
print(f"nearly_equal: {int(mask.sum())} / {mask.size} samples within tolerance")

## 11. Cross-compilation and arch-aware kernels

Two pieces fall out of the §5 caching design and the §2 kernel
library factories:

1. **Cross-compile.** `target_arch` is part of the cache hash, and
   the hash now tracks the device set via `iron.set_current_device(...)`
   rather than only the XRT-detected hardware.  So you can build a
   Strix binary on a Phoenix machine (or vice versa) by switching the
   active device before calling
   `.compile()` — the artifact lands in its own per-arch cache dir,
   no collision with binaries for the actual hardware.

2. **Arch-aware kernel introspection.** Some kernel-library
   factories compile to different MMUL geometries on different NPU
   generations (AIE2 i16/i16 is `4x4x4`, AIE2P i16/i16 is `4x4x8`).
   `kernels.mm(...)` exposes the active geometry as `.mac_dims =
   (r, s, t)` so portable designs can drive their DMA layout
   transforms from the kernel itself instead of hardcoding for one
   arch.

Demo:

In [ ]:
from pathlib import Path

from aie.iron.device import NPU1Col1, NPU2Col1
from aie.utils.hostruntime import set_current_device


# Same generator, two arches → two real xclbins in two cache dirs.
@jit
def cross_compile_demo(x: In, y: Out, *, N: Compile[int]):
    line_ty = np.ndarray[(N,), np.dtype[np.uint8]]
    of_in = ObjectFifo(line_ty, name="xc_in")
    of_out = ObjectFifo(line_ty, name="xc_out")

    def core_fn(of_in, of_out):
        elem_in = of_in.acquire(1)
        elem_out = of_out.acquire(1)
        for i in range_(N):
            elem_out[i] = elem_in[i]
        of_in.release(1)
        of_out.release(1)

    worker = Worker(core_fn, [of_in.cons(), of_out.prod()])
    rt = Runtime()
    with rt.sequence(line_ty, line_ty) as (a, b):
        rt.start(worker)
        rt.fill(of_in.prod(), a)
        rt.drain(of_out.cons(), b, wait=True)
    return Program(iron.get_current_device(), rt).resolve_program()


print("=== cross-compile: same design, both arches → real xclbins ===")
for label, dev_cls in [("Phoenix (npu1)", NPU1Col1), ("Strix   (npu2)", NPU2Col1)]:
    set_current_device(dev_cls())
    xclbin, _insts = cross_compile_demo.specialize(N=4096).compile()
    print(f"  {label}: {Path(xclbin).parent.name}/  ({Path(xclbin).stat().st_size} B)")

print()
print("=== kernels.mm(int16, int16) MMUL geometry per arch ===")
for label, dev_cls in [("Phoenix (npu1)", NPU1Col1), ("Strix   (npu2)", NPU2Col1)]:
    set_current_device(dev_cls())
    mm = kernels.mm(
        dim_m=64,
        dim_k=64,
        dim_n=64,
        input_dtype=np.int16,
        output_dtype=np.int16,
    )
    print(f"  {label}: mac_dims = {mm.mac_dims}")

# Restore Phoenix as the active device for the rest of the notebook.
set_current_device(NPU1Col1())
print("\n(restored active device to Phoenix)")

## 12. Lower-level IRON primitives — peers of `ObjectFifo`

When `ObjectFifo` is the wrong shape for what a design wants to express
(per-tile DMA programs with explicit BD chains, hand-wired locks,
multi-direction routes that don't fit the producer/consumer model),
IRON exposes the underlying pieces as first-class API:

| Primitive | What it wraps |
|-----------|---------------|
| `Buffer` | `aie.buffer` on a specific tile (covered above; can be cross-tile via `Worker.fn_args`) |
| `Lock(tile, lock_id, init, name)` | `aie.lock` with explicit id + init count |
| `Flow(src, src_port, src_channel, dst, dst_port, dst_channel)` | `aie.flow` — an explicit routing pair (the analogue of an ObjectFifo's stream switch wiring) |
| `TileDma(tile, channels=[DmaChannel(direction, channel, bds=[Bd(...)])])` | `aie.{mem,memtile_dma,shim_dma}` body — a per-tile DMA program |
| `Bd(buffer, offset, length, acquires=[Acquire(lock, value)], releases=[Release(lock, value)], next=...)` | One BD in a chain (`aie.dma_bd` + lock use ops) |

Hooked into the `Runtime` via `rt.add_flow(...)`, `rt.add_lock(...)`,
`rt.add_tile_dma(...)`.  Pair with `rt.inline_ops(fn, [args])` to emit
raw `npu_*` ops (`npu_writebd`, `npu_address_patch`, `npu_push_queue`,
`npu_sync`) into the host-side runtime sequence.

The canonical demo is
`programming_examples/basic/chaining_channels/chaining_channels.py` —
a chained shim → memtile → compute DMA with manual `npu_writebd` calls
in the runtime sequence.  Below: `as_mlir()` for that design to show
that the structural ops (Flow / Lock / per-tile DMA bodies) lower
side-by-side with the host-side raw BD writes.

In [ ]:
# Import the chaining_channels @iron.jit design (sibling repo dir).
import re
import sys

_CC_DIR = Path("..") / "programming_examples" / "basic" / "chaining_channels"
sys.path.insert(0, str(_CC_DIR.resolve()))
try:
    from chaining_channels import chaining_channels
    mlir = chaining_channels.as_mlir(length_bytes=1024, col=0)
finally:
    sys.path.pop(0)

# Show the structural side: Flow / Lock / per-tile DMA bodies + the
# host-side npu.writebd block, side by side.  SSA-binding prefixes (e.g.
# `%memtile_dma_0_1 = aie.memtile_dma(...)`) are stripped for matching.
def _strip_ssa(line: str) -> str:
    return re.sub(r"^%\S+\s*=\s*", "", line.strip())


buckets = {"flows": [], "locks": [], "tile DMA bodies": [], "npu.writebd lines": []}
for line in mlir.splitlines():
    s = _strip_ssa(line)
    if s.startswith("aie.flow("):
        buckets["flows"].append(s[:110])
    elif s.startswith("aie.lock("):
        buckets["locks"].append(s[:110])
    elif s.startswith(("aie.mem(", "aie.memtile_dma(", "aie.shim_dma(")):
        buckets["tile DMA bodies"].append(s[:110])
    elif s.startswith("aiex.npu.writebd"):
        buckets["npu.writebd lines"].append(s[:90] + " …")

for label, lines in buckets.items():
    print(f"=== {label} ({len(lines)}) ===")
    for line in lines[:3]:
        print("  " + line)
    if len(lines) > 3:
        print(f"  ... +{len(lines) - 3} more")
    print()

## 13. Putting it all together — full case studies

Single-file `@iron.jit` designs that exercise the patterns above
end-to-end:

| Design | Path | Patterns covered |
|--------|------|------------------|
| Passthrough kernel + trace | `programming_examples/basic/passthrough_kernel/passthrough_kernel.py` | §1, §8 |
| Vector exp (LUT-backed)    | `programming_examples/basic/vector_exp/vector_exp.py` | §2 |
| GEMM (parametrized AOT)    | `programming_examples/getting_started/03_matrix_multiplication_single_core/matrix_multiplication_single_core.py` | §2 |
| Vector reduce, vector–vector add/mul, vector–scalar mul | `programming_examples/basic/vector_*/` | §1, §2 |
| ResNet conv2_x | `programming_examples/ml/resnet/layers_conv2_x/resnet.py` | §2 shared-buffer factories |
| MobileNet (full BN backbone) | `programming_examples/ml/mobilenet/aie2_mobilenet_iron.py` | §2 `kernels.bn_*` family, multi-column composition |
| Matmul (whole_array, 4xN_cols cascade-free)        | `programming_examples/basic/matrix_multiplication/whole_array/whole_array.py` | §2, §4 |
| Matmul (single_core, with `--trace`)               | `programming_examples/basic/matrix_multiplication/single_core/single_core.py` | §2, §8 |
| Matmul (matrix_vector, scalar mv kernel)           | `programming_examples/basic/matrix_multiplication/matrix_vector/matrix_vector.py` | §2 |
| Matmul (cascade, HW cascade-stream accumulation)   | `programming_examples/basic/matrix_multiplication/cascade/cascade.py` | §2, §4 |
| Eltwise unary (`relu`/`silu`/`gelu` dispatcher)    | `programming_examples/ml/eltwise_unary/eltwise_unary.py` | §2 `transform_parallel` |
| Magika group2 (cross-tile LUT + direct stream)     | `programming_examples/ml/magika/group2.py` | §1 cross-tile Buffer + `aie_stream` |
| Chaining channels (hand-wired DMA)                 | `programming_examples/basic/chaining_channels/chaining_channels.py` | §12 lower-level primitives + `rt.inline_ops` |

The 4 matmul siblings demonstrate the new patterns end-to-end: `kernels.mm/.zero`,
`kernels.cascade_mm/.get_only/.put_only/.put_get/.zero`, `aiecc_flags=[...]`,
compile-only mode via `--xclbin-path`/`--insts-path`, and `CascadeFlow` for
hardware cascade streams.

The resnet design is the most complete demo of shared-buffer kernel
factories + multi-DMA fan-out: it streams weights to 3 columns at
different sizes and uses both
`conv2dk1_skip_init(skip_input_channels=...)` and
`conv2dk3(weight_output_channels=...)`.

MobileNet (#3059) scales the same kernel-library pattern to a full ML
backbone: bn_conv2dk1 variants (`_i8`, `_relu`, `_skip`), bn_conv2dk3
(stride 1+2, plus `_dw` depthwise), running across a multi-column
cascade composition.

## 14. Deferred follow-ups

One feature-shaped gap surfaced during this walkthrough that's still
scoped for future work:

- **`Rtp[T]` annotation for `@iron.jit`** — a first-class runtime scalar
  parameter that gets forwarded into the `aie.runtime_sequence` as a
  non-memref typed block arg, with no recompile per value.  Today the
  only way to pass a non-tensor per-call value is `Compile[T]` (which
  recompiles on change) or a 1-element tensor (the
  `vector_scalar_mul` pattern).  Guard 1-C in `python/utils/jit.py`
  rejects unannotated scalar params *with defaults* precisely to keep
  this gap from becoming a silent footgun: the default would be baked
  into the kernel and per-call overrides silently ignored.

## Wrap-up

Every cell here ran on a live Ryzen AI NPU.  The device
auto-detection in §0 picks NPU1 / NPU2 from the attached hardware,
so the notebook works unchanged on Phoenix and Strix.